<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

# Module 2B: *Spatial Join Reservoirs*
##### Version Number: 4.0
---
### Contents
> *Define Geometries*\
> *Spatial Join of Nearest Sampling Locations with Reservoir Datasets*\
> *Export File*
---
### Notes
- Integrate reservoir readings with sampling grid data via a buffered spatial join.
### Inputs
- Daily Weather Readings - `samples_weather_NDVI.csv` 
- Reservoir Data - `reservoirs.csv` (built in appendix H) 
---
### Outputs  
- `samples_res.csv` Sampling grid dataset merged with buffered reservoir readings.
---
### User Created Dependencies  

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_utils import *
from src.model_utils import *
from src.plot_utils import *

---
### Third Party Dependencies

In [2]:
# Data handling
import pandas as pd
import numpy as np
import geopandas as gpd
import datetime as dt

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
from shapely.geometry import Point, Polygon

---

### Data Loading

In [3]:
samples = pd.read_csv("../data/processed/samples_weather_NDVI.csv")
reservoirs = pd.read_csv("../data/processed/reservoirs.csv")

In [4]:
samples = samples.sort_values(
    by=["grid_id", "Date"]
)

In [5]:
samples['Date'] = pd.to_datetime(samples['Date'])
reservoirs['Date'] = pd.to_datetime(reservoirs['Date'])

### Subset main dataset to speed up spatial join.

In [6]:
keep = ['centroid_easting','centroid_northing','Date','grid_id']
join_samples = samples[keep].copy()

### Add geometry to sampling grid and reservoir datasets

In [7]:
join_samples['geometry'] = [Point(xy) for xy in zip(join_samples ['centroid_easting'], join_samples ['centroid_northing'])]
samples_gdf = gpd.GeoDataFrame(join_samples, geometry='geometry', crs="EPSG:3310")

In [8]:
reservoirs['geometry'] = [Point(xy) for xy in zip(reservoirs['Longitude'], reservoirs['Latitude'])]
reservoir_gdf = gpd.GeoDataFrame(reservoirs, geometry='geometry', crs="EPSG:4326")

## Project to Equal area projection for more accuracy
reservoir_gdf_projected = reservoir_gdf.to_crs(3310)

### Define default buffer size

**Assumption:** Ideally, the buffer would be applied to the centroid points og the sampling grid. However, this would be computationally very heavy. The operation creates a buffer around each reservoir the size of a grid, 46Km x 46 Km. 

In [9]:
# Square Buffer radius
buffer_size = 23000

### Spatial join sampling grid with reservoirs

#### Main loop of Spatial Join algorithm
- Chunk datasets by year for faster operation
- Loop through all dates in samples dataset
- Save all reservoirs and grid centroids associated with the current date
    - If no reservoir readings on current data, move to next date
- Create a **square buffer** around each around each reservoir
- Conduct **intersection spatial join** of sampling centroids and buffered reservoirs
- Aggregate in case of multiple reservoirs in a buffered range
    - `reservoir_count` Amount of reservoirs within the 46K x 46K grid
    - `total_reservoir_level` Sum of all reservoir readings within the 46K x 46K grid
    - `stations_missing_levels` Count of how many reservoirs are missing readings
- Assign samples to new dataframe to join back with main dataset

In [10]:
def add_reservoir_features_for_year(
    samples_gdf,
    reservoir_gdf_projected,
    year,
    buffer_size,
    square_buffer_fn,
):
    # Limit both datasets to year chunk
    samples_year = samples_gdf[samples_gdf["Date"].dt.year == year]
    reservoirs_year = reservoir_gdf_projected[
        reservoir_gdf_projected["Date"].dt.year == year
    ]

    # If no reservoirs or samples present in this year, skip
    if samples_year.empty or reservoirs_year.empty:
        return samples_gdf

    
    for dt in samples_year["Date"].unique():
        
        # Subset all reservoirs this date
        res_today = reservoirs_year[reservoirs_year["Date"] == dt]
        if res_today.empty:
            continue

        # Subset all sampling points on this date
        samples_today = samples_year[samples_year["Date"] == dt]
        if samples_today.empty:
            continue

        # Buffer reservoirs
        res_buffered = res_today.copy()
        res_buffered["geometry"] = res_buffered.geometry.apply(
            lambda p: square_buffer_fn(p, buffer_size)
        )

        # Spatial join reservoirs to samples dataframe
        joined = gpd.sjoin(
            samples_today,
            res_buffered,
            how="left",
            predicate="intersects"
        )

        # Missing data flag, orignially 0, change to 1 to maintain a running count
        # if there are multiple reservoirs in a sampling grid
        joined["level_missing"] = 1 - joined["Reservoir Level_has_data"]

        # Aggregate per grid cell
        grouped = (
            joined
            .groupby("grid_id")
            .agg(
                reservoir_count=("Station_ID", "count"),
                total_reservoir_level=("Reservoir Level", "sum"),
                stations_missing_levels=("level_missing", "sum"),
            )
            .fillna(0)
        )

        # Write results back to main dataframe
        mask = samples_gdf["Date"] == dt
        samples_gdf.loc[mask, "reservoir_count"] = (
            samples_gdf.loc[mask, "grid_id"]
            .map(grouped["reservoir_count"])
            .fillna(0)
        )
        samples_gdf.loc[mask, "total_reservoir_level"] = (
            samples_gdf.loc[mask, "grid_id"]
            .map(grouped["total_reservoir_level"])
            .fillna(0)
        )
        samples_gdf.loc[mask, "stations_missing_levels"] = (
            samples_gdf.loc[mask, "grid_id"]
            .map(grouped["stations_missing_levels"])
            .fillna(0)
        )

    return samples_gdf

In [11]:
for yr in sorted(samples_gdf["Date"].dt.year.unique()):
    samples_gdf = add_reservoir_features_for_year(
        samples_gdf=samples_gdf,
        reservoir_gdf_projected=reservoir_gdf_projected,
        year=yr,
        buffer_size=buffer_size,
        square_buffer_fn=square_buffer,
    )

In [12]:
post_merge_check(samples_gdf, join_samples)

Premerged shape:  (608880, 5)
Merged shape:  (608880, 8)


Duplicates before merge:  0


Duplicates after merge:  0
NA values before merge:  0
NA values after merge:  0


In [13]:
samples_gdf = samples_gdf.drop(columns=['centroid_easting','centroid_northing','geometry'])

### Rejoin with main dataset

In [14]:
# Merge on BOTH station and date
samples_res = samples.merge(
    samples_gdf,
    on=['grid_id','Date'],
    how='left'
)

In [15]:
post_merge_check(samples_res, samples)

Premerged shape:  (608880, 65)
Merged shape:  (608880, 68)


Duplicates before merge:  0


Duplicates after merge:  0
NA values before merge:  0


NA values after merge:  0


In [16]:
samples_res.isna().sum()

Sample_ID                         0
Date                              0
Burning Index                     0
Energy Release Component          0
Actual Evapotranspiration         0
                                 ..
NDVI_mean_difference              0
NDVI_mean_difference_has_value    0
reservoir_count                   0
total_reservoir_level             0
stations_missing_levels           0
Length: 68, dtype: int64

## Export File

In [17]:
samples_res.to_csv('../data/processed/samples_res.csv',index=False)
print("All datasets saved successfully to ../data/processed/")

All datasets saved successfully to ../data/processed/
